#Reading data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Reading y and gain

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from concurrent.futures import ProcessPoolExecutor

In [ ]:
def process_file(timestamp, x_dir, y_dir):
    vm_path = os.path.join(x_dir, timestamp, 'x_virtual_machines.json')
    pm_path = os.path.join(x_dir, timestamp, 'x_physical_machines.json')
    y_timestamp = "y" + timestamp[1:] + '.csv'
    output_path = os.path.join(y_dir, y_timestamp)

    if os.path.exists(vm_path) and os.path.exists(pm_path) and os.path.exists(output_path):
        with open(vm_path, 'r') as vm_file, open(pm_path, 'r') as pm_file:
            vms = json.load(vm_file)
            pms = json.load(pm_file)

        output_df = pd.read_csv(output_path)
        mask = ~output_df.astype(str).iloc[:, :2].apply(lambda col: col.str.contains('Acik|Kapali')).any(axis=1)
        filtered_df = output_df.loc[mask].iloc[:, :2]

        print(f'Processed file: {timestamp}')
        return (vms, pms), filtered_df.values.tolist()
    return None

def load_training_data(data_dir):
    x_dir = os.path.join(data_dir, 'x')
    y_dir = os.path.join(data_dir, 'y')
    timestamps = os.listdir(x_dir)
    inputs, outputs = [], []

    with ProcessPoolExecutor() as executor:
        results = executor.map(process_file, timestamps, [x_dir]*len(timestamps), [y_dir]*len(timestamps))

    for result in results:
        if result is not None:
            input_data, output_data = result
            inputs.append(input_data)
            outputs.append(output_data)

    return inputs, outputs

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Helper function to find gain JSON file inside possible subfolders
def find_gain_json_file(gain_folder, expected_filename):
    for entry in os.listdir(gain_folder):
        full_path = os.path.join(gain_folder, entry)
        if os.path.isdir(full_path):
            candidate = os.path.join(full_path, expected_filename)
            if os.path.isfile(candidate):
                return candidate
    return None

# Set your main testing folder path here (update to your exact path!)
testing_folder_path = "/content/drive/MyDrive/LLM/testing_data after weights update"

old_method_data = []
new_method_data = []

for sample in os.listdir(testing_folder_path):
    try:
        sample_number = int(sample.split('_')[1])
    except:
        print(f"Skipping non-sample folder/file: {sample}")
        continue

    if sample_number < 907:
        continue  # Skip before 907

    sample_path = os.path.join(testing_folder_path, sample)

    new_method_path = os.path.join(sample_path, "new_method")
    old_method_path = os.path.join(sample_path, "old_method")

    if not (os.path.exists(new_method_path) and os.path.exists(old_method_path)):
        print(f"Skipping sample {sample} as new_method or old_method folder missing.")
        continue

    new_y_folder = os.path.join(new_method_path, "y")
    old_y_folder = os.path.join(old_method_path, "y")

    new_y_files = os.listdir(new_y_folder)
    old_y_files = os.listdir(old_y_folder)

    new_y_path = os.path.join(new_y_folder, new_y_files[0])
    old_y_path = os.path.join(old_y_folder, old_y_files[0])

    df_new_y = pd.read_csv(new_y_path)
    df_old_y = pd.read_csv(old_y_path)

    if len(df_new_y) != len(df_old_y):
        print(f"Skipping sample {sample} due to row count mismatch in y files.")
        continue

    new_gain_folder = os.path.join(new_method_path, "gain")
    old_gain_folder = os.path.join(old_method_path, "gain")

    new_gain_path = find_gain_json_file(new_gain_folder, "new_method_gain.json")
    old_gain_path = find_gain_json_file(old_gain_folder, "old_method_gain.json")

    if new_gain_path is None or old_gain_path is None:
        print(f"Skipping sample {sample} as gain JSON files were not found.")
        continue

    with open(new_gain_path, "r") as f:
        new_gain = json.load(f)
    with open(old_gain_path, "r") as f:
        old_gain = json.load(f)

    new_method_data.append({
        "sample": sample,
        "past_power": new_gain["past_power"],
        "cur_power": new_gain["cur_power"],
        "prop_power": new_gain["prop_power"],
        "prop_ratio": new_gain["prop_ratio"],
        "actual_ratio": new_gain["actual_ratio"],
        "val_ratio": new_gain["val_ratio"],
        "val_difference": new_gain["val_difference"],
        "score": new_gain["score"]
    })

    old_method_data.append({
        "sample": sample,
        "past_power": old_gain["past_power"],
        "cur_power": old_gain["cur_power"],
        "prop_power": old_gain["prop_power"],
        "prop_ratio": old_gain["prop_ratio"],
        "actual_ratio": old_gain["actual_ratio"],
        "val_ratio": old_gain["val_ratio"],
        "val_difference": old_gain["val_difference"],
        "score": old_gain["score"]
    })

df_new = pd.DataFrame(new_method_data)
df_old = pd.DataFrame(old_method_data)

print("✅ Y + Gain data loaded successfully.")

Skipping sample sample_936 due to row count mismatch in y files.
Skipping sample sample_940 due to row count mismatch in y files.
Skipping sample sample_934 due to row count mismatch in y files.
Skipping sample sample_938 due to row count mismatch in y files.
Skipping sample sample_937 due to row count mismatch in y files.
Skipping sample sample_929 due to row count mismatch in y files.
Skipping sample sample_930 due to row count mismatch in y files.
Skipping sample sample_915 due to row count mismatch in y files.
Skipping sample sample_920 due to row count mismatch in y files.
Skipping sample sample_912 due to row count mismatch in y files.
Skipping sample sample_914 due to row count mismatch in y files.
Skipping sample sample_916 as new_method or old_method folder missing.
Skipping sample sample_907 due to row count mismatch in y files.
Skipping sample sample_908 due to row count mismatch in y files.
✅ Y + Gain data loaded successfully.


In [ ]:
print(f"Number of loaded samples: {len(df_new)}")

Number of loaded samples: 20


##Reading x

In [ ]:
import os
import json
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

def process_file(args):
    timestamp, x_dir, y_dir = args
    vm_path = os.path.join(x_dir, timestamp, 'x_virtual_machines.json')
    pm_path = os.path.join(x_dir, timestamp, 'x_physical_machines.json')  # fixed typo here
    y_timestamp = "y" + timestamp[1:] + '.csv'
    output_path = os.path.join(y_dir, y_timestamp)

    if os.path.exists(vm_path) and os.path.exists(pm_path) and os.path.exists(output_path):
        with open(vm_path, 'r') as vm_file, open(pm_path, 'r') as pm_file:
            vms = json.load(vm_file)
            pms = json.load(pm_file)

        output_df = pd.read_csv(output_path)
        mask = ~output_df.astype(str).iloc[:, :2].apply(lambda col: col.str.contains('Acik|Kapali')).any(axis=1)
        filtered_df = output_df.loc[mask].iloc[:, :2]

        print(f'Processed file: {timestamp}')
        return (vms, pms), filtered_df.values.tolist()
    else:
        print(f'Missing files for timestamp: {timestamp}')
    return None

def load_training_data_for_sample(sample_path):
    x_dir = os.path.join(sample_path, 'new_method', 'x')
    y_dir = os.path.join(sample_path, 'new_method', 'y')

    if not (os.path.exists(x_dir) and os.path.exists(y_dir)):
        print(f"Skipping sample {sample_path} as x or y folder missing")
        return [], []

    timestamps = os.listdir(x_dir)
    inputs, outputs = [], []

    args_list = [(timestamp, x_dir, y_dir) for timestamp in timestamps]

    with ProcessPoolExecutor() as executor:
        results = executor.map(process_file, args_list)

    for result in results:
        if result is not None:
            input_data, output_data = result
            inputs.append(input_data)
            outputs.append(output_data)

    return inputs, outputs

# Main loop to load data from all samples (you can filter samples here if needed)
testing_folder_path = "/content/drive/MyDrive/LLM/testing_data after weights update"

all_inputs = []
all_outputs = []

for sample in os.listdir(testing_folder_path):
    sample_path = os.path.join(testing_folder_path, sample)
    if not os.path.isdir(sample_path):
        continue
    print(f"Loading data for sample: {sample}")
    inputs, outputs = load_training_data_for_sample(sample_path)
    all_inputs.extend(inputs) ### why extend ?
    all_outputs.extend(outputs)

print(f"✅ X data loaded successfully.")
print(f"Number of training samples: {len(all_inputs)}")

'''
x_virtual_machines.json → part of input
x_physical_machines.json → part of input

y_xxxx.csv → used as output
'''

Loading data for sample: sample_936
Processed file: x_2025_03_11_08_52
Loading data for sample: sample_940
Processed file: x_2025_03_11_09_58
Loading data for sample: sample_934
Processed file: x_2025_03_11_08_16
Loading data for sample: sample_931
Processed file: x_2025_03_11_07_18
Loading data for sample: sample_933
Processed file: x_2025_03_11_07_59
Loading data for sample: sample_939
Processed file: x_2025_03_11_09_43
Loading data for sample: sample_932
Processed file: x_2025_03_11_07_40
Loading data for sample: sample_938
Processed file: x_2025_03_11_09_25
Loading data for sample: sample_935
Processed file: x_2025_03_11_08_35
Loading data for sample: sample_937
Processed file: x_2025_03_11_09_09
Loading data for sample: sample_927
Processed file: x_2025_03_11_06_06
Loading data for sample: sample_929
Processed file: x_2025_03_11_06_42
Loading data for sample: sample_926
Processed file: x_2025_03_11_05_48
Loading data for sample: sample_928
Processed file: x_2025_03_11_06_25
Loadin

'\nx_virtual_machines.json → part of input\nx_physical_machines.json → part of input\n\ny_xxxx.csv → used as output\n'

In [ ]:
( type(all_inputs[0]),type(all_outputs[0]))
( type(all_inputs),type(all_outputs) , )

(list, list)

##Loaded samples

In [ ]:
import os
import json
import pandas as pd

def load_new_method_data(sample_path):
    new_method_path = os.path.join(sample_path, "new_method")
    x_base = os.path.join(new_method_path, "x")
    y_base = os.path.join(new_method_path, "y")
    gain_base = os.path.join(new_method_path, "gain")

    if not os.path.exists(x_base):
        print(f"Skipping sample {os.path.basename(sample_path)}: missing new_method/x folder.")
        return []

    data = []

    timestamps = [folder for folder in os.listdir(x_base) if folder.startswith("x_")]

    for ts_folder in timestamps:
        ts = ts_folder[2:]  # remove "x_" prefix

        x_folder_path = os.path.join(x_base, ts_folder)
        vm_path = os.path.join(x_folder_path, "x_virtual_machines.json")
        pm_path = os.path.join(x_folder_path, "x_physical_machines.json")

        y_file_name = f"y_{ts}.csv"
        y_path = os.path.join(y_base, y_file_name)

        gain_folder_name = f"gain_{ts}"
        gain_path = os.path.join(gain_base, gain_folder_name, "new_method_gain.json")

        # Check required files exist
        if not (os.path.exists(vm_path) and os.path.exists(pm_path) and os.path.exists(y_path) and os.path.exists(gain_path)):
            print(f"Skipping timestamp {ts} in sample {os.path.basename(sample_path)} due to missing files.")
            continue

        with open(vm_path, "r") as f:
            vms = json.load(f)
        with open(pm_path, "r") as f:
            pms = json.load(f)

        df_alloc = pd.read_csv(y_path)
        allocation = df_alloc.iloc[:, :2].values.tolist()

        with open(gain_path, "r") as f:
            gain = json.load(f)

        data.append((vms, pms, allocation, gain))

    return data

# Base directory where your samples folder is
base_dir = "/content/drive/MyDrive/LLM/testing_data after weights update"

all_samples = [folder for folder in os.listdir(base_dir) if folder.startswith("sample_")]

all_data = []

for sample in all_samples:
    sample_path = os.path.join(base_dir, sample)
    sample_data = load_new_method_data(sample_path)
    if sample_data:
        print(f"Loaded {len(sample_data)} timestamps from {sample}")
        all_data.extend(sample_data)
    else:
        print(f"No valid data found in {sample}")

print(f"\nTotal valid timestamps loaded from all samples: {len(all_data)}")

Loaded 1 timestamps from sample_936
Loaded 1 timestamps from sample_940
Loaded 1 timestamps from sample_934
Loaded 1 timestamps from sample_931
Loaded 1 timestamps from sample_933
Loaded 1 timestamps from sample_939
Loaded 1 timestamps from sample_932
Loaded 1 timestamps from sample_938
Loaded 1 timestamps from sample_935
Loaded 1 timestamps from sample_937
Loaded 1 timestamps from sample_927
Loaded 1 timestamps from sample_929
Loaded 1 timestamps from sample_926
Loaded 1 timestamps from sample_928
Loaded 1 timestamps from sample_923
Loaded 1 timestamps from sample_930
Loaded 1 timestamps from sample_925
Loaded 1 timestamps from sample_922
Loaded 1 timestamps from sample_921
Loaded 1 timestamps from sample_924
Loaded 1 timestamps from sample_917
Loaded 1 timestamps from sample_911
Loaded 1 timestamps from sample_918
Loaded 1 timestamps from sample_915
Loaded 1 timestamps from sample_913
Loaded 1 timestamps from sample_920
Loaded 1 timestamps from sample_912
Loaded 1 timestamps from sam

In [ ]:
import os
import json
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

def process_timestamp(timestamp, x_dir, y_dir):
    """
    Process a single timestamp within a sample to load VM, PM, and allocation (y) data.

    Args:
        timestamp (str): Folder name (timestamp) inside the x directory.
        x_dir (str): Path to x directory (contains VM and PM files).
        y_dir (str): Path to y directory (contains allocation CSV).

    Returns:
        dict or None: Dictionary with 'vms', 'pms', 'allocation' keys or None if files are missing.
    """
    vm_path = os.path.join(x_dir, timestamp, 'x_virtual_machines.json')
    pm_path = os.path.join(x_dir, timestamp, 'x_physical_machines.json')
    y_filename = "y" + timestamp[1:] + '.csv'
    y_path = os.path.join(y_dir, y_filename)

    if os.path.exists(vm_path) and os.path.exists(pm_path) and os.path.exists(y_path):
        with open(vm_path, 'r') as f_vm, open(pm_path, 'r') as f_pm:
            vms = json.load(f_vm)
            pms = json.load(f_pm)

        df = pd.read_csv(y_path)
        mask = ~df.astype(str).iloc[:, :2].apply(lambda col: col.str.contains('Acik|Kapali')).any(axis=1)
        allocations = df.loc[mask].iloc[:, :2].values.tolist()

        print(f"✅ Processed timestamp: {timestamp}")
        return {
            "vms": vms,
            "pms": pms,
            "allocation": allocations
        }

    print(f"⚠️ Skipped timestamp: {timestamp} (missing files)")
    return None

def load_sample_data(sample_path):
    """
    Load data for the 'new_method' from a single sample folder.

    Args:
        sample_path (str): Path to a single sample folder (e.g., 'sample_907').

    Returns:
        list: List of dicts with 'vms', 'pms', and 'allocation' keys for the timestamps within this sample.
    """
    new_method_path = os.path.join(sample_path, "new_method")
    x_dir = os.path.join(new_method_path, "x")
    y_dir = os.path.join(new_method_path, "y")

    if not (os.path.exists(x_dir) and os.path.exists(y_dir)):
        print(f"Skipping sample {os.path.basename(sample_path)}: missing new_method/x or new_method/y folder.")
        return []

    timestamps = os.listdir(x_dir)
    results = []

    args_list = [(timestamp, x_dir, y_dir) for timestamp in timestamps]

    with ProcessPoolExecutor() as executor:
        futures = executor.map(process_timestamp, timestamps, [x_dir] * len(timestamps), [y_dir] * len(timestamps))

    for item in futures:
        if item:
            results.append(item)

    return results


def save_as_json(data, output_path="structured_dataset.json"):
    """
    Save the structured dataset to a JSON file.

    Args:
        data (list): List of samples with 'vms', 'pms', and 'allocation'.
        output_path (str): Output file name.
    """
    with open(output_path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"\n✅ Structured JSON saved to: {output_path}")

# ----------------------
# 🚀 Run the pipeline
# ----------------------

# Base directory where your samples folder is
testing_folder_path = "/content/drive/MyDrive/LLM/testing_data after weights update"

all_samples_data = []

# Iterate through each sample folder
for sample_name in os.listdir(testing_folder_path):
    sample_path = os.path.join(testing_folder_path, sample_name)
    if os.path.isdir(sample_path) and sample_name.startswith("sample_"):
        print(f"Processing sample: {sample_name}")
        sample_data = load_sample_data(sample_path)
        all_samples_data.extend(sample_data)

print(f"\nTotal valid timestamps loaded from all samples: {len(all_samples_data)}")

save_as_json(all_samples_data, "structured_dataset.json")

Processing sample: sample_936
✅ Processed timestamp: x_2025_03_11_08_52
Processing sample: sample_940
✅ Processed timestamp: x_2025_03_11_09_58
Processing sample: sample_934
✅ Processed timestamp: x_2025_03_11_08_16
Processing sample: sample_931
✅ Processed timestamp: x_2025_03_11_07_18
Processing sample: sample_933
✅ Processed timestamp: x_2025_03_11_07_59
Processing sample: sample_939
✅ Processed timestamp: x_2025_03_11_09_43
Processing sample: sample_932
✅ Processed timestamp: x_2025_03_11_07_40
Processing sample: sample_938
✅ Processed timestamp: x_2025_03_11_09_25
Processing sample: sample_935
✅ Processed timestamp: x_2025_03_11_08_35
Processing sample: sample_937
✅ Processed timestamp: x_2025_03_11_09_09
Processing sample: sample_927
✅ Processed timestamp: x_2025_03_11_06_06
Processing sample: sample_929
✅ Processed timestamp: x_2025_03_11_06_42
Processing sample: sample_926
✅ Processed timestamp: x_2025_03_11_05_48
Processing sample: sample_928
✅ Processed timestamp: x_2025_03_1

##Run-time Reading

In [ ]:
training_samples = []

for vms, pms, allocation, gain in all_data:
    # Build instruction string (same as before)
    instruction = f"""
You are an AI expert in cloud computing and virtualization. Your task is to analyze the given Virtual Machines (VMs) and Physical Machines (PMs) and suggest the best VM migrations to optimize performance, minimize energy consumption, and balance workload distribution.

Your decision-making must be based on the following key factors:
- *CPU Utilization*: Ensure that PMs are not overloaded and CPU resources are balanced.
- *RAM Utilization*: Avoid memory bottlenecks by distributing VMs efficiently.
- *Disk Storage Usage*: Make sure PMs have enough storage to accommodate VM migrations.
- *Energy Consumption*: Prefer migrations that reduce overall power usage while maintaining performance.

### Important Constraints:
- *Every virtual machine (VM) MUST be assigned to a physical machine (PM).* No VM should be left unallocated DON'T EVER FORGET THAT.
- *Ensure the allocation follows the given PM capacity constraints for CPU, RAM, disk storage, and energy consumption.*
- *Balance the workload to avoid overloading specific PMs while keeping overall energy consumption low.*

Here are the VM configurations:
{json.dumps(vms, indent=2)}

Here are the PM configurations:
{json.dumps(pms, indent=2)}

Your task is to determine the best target PM for each VM while ensuring that all VMs are allocated.

Provide a list of VMs that need to be migrated and the best target PMs for them *without saying any further words. **Only output a valid Python dictionary* where:
- *Keys* are the VM names.
- *Values* are the best target PM names.
""".strip()

    response_dict = {vm: pm for vm, pm in allocation}

    training_samples.append({
        "instruction": instruction,
        "output": json.dumps(response_dict, indent=2)
    })

In [ ]:
from datasets import Dataset

# Create Hugging Face Dataset directly from the (((list))
dataset = Dataset.from_list(training_samples)

print(dataset[0])

{'instruction': 'You are an AI expert in cloud computing and virtualization. Your task is to analyze the given Virtual Machines (VMs) and Physical Machines (PMs) and suggest the best VM migrations to optimize performance, minimize energy consumption, and balance workload distribution.\n\nYour decision-making must be based on the following key factors:\n- *CPU Utilization*: Ensure that PMs are not overloaded and CPU resources are balanced.\n- *RAM Utilization*: Avoid memory bottlenecks by distributing VMs efficiently.\n- *Disk Storage Usage*: Make sure PMs have enough storage to accommodate VM migrations.\n- *Energy Consumption*: Prefer migrations that reduce overall power usage while maintaining performance.\n\n### Important Constraints:\n- *Every virtual machine (VM) MUST be assigned to a physical machine (PM).* No VM should be left unallocated DON\'T EVER FORGET THAT.\n- *Ensure the allocation follows the given PM capacity constraints for CPU, RAM, disk storage, and energy consumptio

#fine-tune

In [ ]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False. ###

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.3: Fast Mistral patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.1.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(training_samples)

In [ ]:
from transformers import AutoTokenizer

base_model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"  # replace with your model
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token  # important for causal LM padding

def tokenize_function(example):
    # Changed 'response' to 'output' to match the dataset structure
    prompt = f"### Instruction:\n{example['instruction']}\n\n### output:\n{example['output']}"
    tokens = tokenizer(
        prompt,
        max_length=512,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    return {
        "input_ids": tokens["input_ids"][0],
        "attention_mask": tokens["attention_mask"][0],
        "labels": tokens["input_ids"][0].clone(),
    }

tokenized_dataset = dataset.map(tokenize_function)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

Map:   0%|          | 0/58 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = tokenized_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 40,  # number of traning steps
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

In [ ]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 58 | Num Epochs = 5 | Total steps = 40
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Step,Training Loss
1,1.323800
2,1.318600
3,1.189500
4,0.997100
5,0.806900
6,0.580900
7,0.371000
8,0.228800
9,0.183100
10,0.141200


Saving the model in the runtime

In [ ]:
model.save_pretrained("lora_model") # Local saving
tokenizer.save_pretrained("lora_model")

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.model',
 'lora_model/added_tokens.json',
 'lora_model/tokenizer.json')

Saving the model in drive.

In [ ]:
from google.colab import drive


# Mount Google Drive
drive.mount('/content/drive')

# Define save paths
save_path = "/content/drive/MyDrive/unsloth_model_300"

# Save the model (assuming 'model' is your PeftModel)
model.save_pretrained(save_path)  # Saves adapter weights and config

# If you want to save the entire model (base model + adapter)
# model.save_pretrained(save_path, save_adapter=True, save_embedding_layers=True)

# Save tokenizer
tokenizer.save_pretrained(save_path)

# Optional: If you need to save the base model separately
# model.base_model.save_pretrained(save_path + "_base")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


('/content/drive/MyDrive/unsloth_model_300/tokenizer_config.json',
 '/content/drive/MyDrive/unsloth_model_300/special_tokens_map.json',
 '/content/drive/MyDrive/unsloth_model_300/chat_template.jinja',
 '/content/drive/MyDrive/unsloth_model_300/tokenizer.model',
 '/content/drive/MyDrive/unsloth_model_300/added_tokens.json',
 '/content/drive/MyDrive/unsloth_model_300/tokenizer.json')

Merging the trained model with the base model

In [ ]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from peft import PeftModel

# === 1. Load base model again (4bit) ===
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",  # this loads base model
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)

# === 2. Load your saved LoRA adapter ===
lora_path = "/content/drive/MyDrive/unsloth_model_300"
model = PeftModel.from_pretrained(model, lora_path)

# === 3. Merge LoRA into the base model ===
model = model.merge_and_unload()

# === 4. Save merged model ===
merged_path = "/content/drive/MyDrive/unsloth_model_300_merged"
model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)

print("✅ Merged model saved to:", merged_path)


==((====))==  Unsloth 2025.7.5: Fast Mistral patching. Transformers: 4.53.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


✅ Merged model saved to: /content/drive/MyDrive/unsloth_model_300_merged


Converting the merge model to ollama

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load the fine-tuned model with device_map="auto"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/drive/MyDrive/unsloth_model_300_merged",  # This is your fine-tuned folder
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
    device_map="auto" # Explicitly set device_map="auto"
)

# Save to GGUF format
# I will keep the quantization_method as 'q4_k_m' as it's a common and efficient choice for Ollama.
model.save_pretrained_gguf("/content/drive/MyDrive/ollama_mistral_unsloth_model", tokenizer, quantization_method="q4_k_m")

==((====))==  Unsloth 2025.7.5: Fast Mistral patching. Transformers: 4.53.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Will load /content/drive/MyDrive/unsloth_model_300_merged as a legacy tokenizer.
Unsloth: You have 1 CPUs. Using `safe_serialization` is 10x slower.
We shall switch to Pytorch saving, which might take 3 minutes and not 30 minutes.
To force `safe_serialization`, set it to `None` instead.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 2.89 out of 12.67 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


 16%|█▌        | 5/32 [00:00<00:00, 47.92it/s]
We will save to Disk and not RAM now.
100%|██████████| 32/32 [04:13<00:00,  7.91s/it]


Unsloth: Saving tokenizer... Done.
Unsloth: Saving /content/drive/MyDrive/ollama_mistral_unsloth_model/pytorch_model-00001-of-00003.bin...
Unsloth: Saving /content/drive/MyDrive/ollama_mistral_unsloth_model/pytorch_model-00002-of-00003.bin...
Unsloth: Saving /content/drive/MyDrive/ollama_mistral_unsloth_model/pytorch_model-00003-of-00003.bin...
Done.


Unsloth: Converting mistral model. Can use fast conversion = True.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: CMAKE detected. Finalizing some steps for installation.
Unsloth: [1] Converting model at /content/drive/MyDrive/ollama_mistral_unsloth_model into f16 GGUF format.
The output location will be /content/drive/MyDrive/ollama_mistral_unsloth_model/unsloth.F16.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: ollama_mistral_unsloth_model
INFO:hf-to-gguf:Model architecture: MistralForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model weight map from 'pytorch_model.bin.index.

For testing

In [ ]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
import torch

# ✅ 1. Load the model the correct way (Unsloth-compatible)
model_path = "lora_model"  # example: "/content/drive/MyDrive/lora_model"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    dtype = None,  # Automatically chooses best dtype
    load_in_4bit = True,  # Match what you used during fine-tuning
)

FastLanguageModel.for_inference(model)  # ✅ This wraps the model to enable .generate()
model.max_seq_length = 2048 # explicitly set max_seq_length here


# ✅ 2. Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device) # Remove this line as it's not supported for 8-bit models